# DeepWhale — Pipeline Runner

End-to-end orchestration of the DeepWhale ML pipeline.
Runs all 6 scripts in sequence and reports status for each step.

| Step | Script | Output |
|------|--------|--------|
| 1 | `data_collection.py` | `data/raw_whale_transactions.csv` |
| 2 | `feature_engineering.py` | `data/address_features.csv` |
| 3 | `labeling.py` | `data/labeled_addresses.csv` |
| 4 | `clustering.py` | `data/clustered_addresses.csv` |
| 5 | `classification.py` | `models/whale_classifier.pkl` |
| 6 | `anomaly_detection.py` | `data/anomaly_scores.csv` |

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

BASE_DIR = Path.cwd()
PYTHON = sys.executable

PIPELINE_STEPS = [
    {"name": "Data Collection",       "script": "data_collection.py",    "output": "data/raw_whale_transactions.csv"},
    {"name": "Feature Engineering",   "script": "feature_engineering.py", "output": "data/address_features.csv"},
    {"name": "Labeling",              "script": "labeling.py",           "output": "data/labeled_addresses.csv"},
    {"name": "Clustering",            "script": "clustering.py",         "output": "data/clustered_addresses.csv"},
    {"name": "Classification",        "script": "classification.py",     "output": "models/whale_classifier.pkl"},
    {"name": "Anomaly Detection",     "script": "anomaly_detection.py",  "output": "data/anomaly_scores.csv"},
]


def run_step(step: dict) -> dict:
    """Run a single pipeline step and return result info."""
    script_path = BASE_DIR / step["script"]
    output_path = BASE_DIR / step["output"]

    print(f"\n{'='*60}")
    print(f"  Step: {step['name']}")
    print(f"  Script: {step['script']}")
    print(f"{'='*60}")

    if not script_path.exists():
        print(f"  SKIPPED — {step['script']} not found")
        return {"status": "skipped", "reason": "script not found"}

    t0 = time.time()
    result = subprocess.run(
        [PYTHON, str(script_path)],
        capture_output=True,
        text=True,
        cwd=str(BASE_DIR),
    )
    elapsed = time.time() - t0

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(f"[stderr]\n{result.stderr}")

    if result.returncode != 0:
        print(f"  FAILED (exit code {result.returncode}) — {elapsed:.1f}s")
        return {"status": "failed", "exit_code": result.returncode, "time": elapsed}

    output_exists = output_path.exists()
    print(f"  OK — {elapsed:.1f}s | output exists: {output_exists}")
    return {"status": "ok", "time": elapsed, "output_exists": output_exists}


print(f"Python: {PYTHON}")
print(f"Working dir: {BASE_DIR}")

## Run Full Pipeline

In [ ]:
results = {}
total_start = time.time()

for step in PIPELINE_STEPS:
    results[step["name"]] = run_step(step)
    if results[step["name"]]["status"] == "failed":
        print(f"\nPipeline stopped at '{step['name']}'. Fix the error above and re-run.")
        break

total_elapsed = time.time() - total_start
print(f"\n{'='*60}")
print(f"  Pipeline finished in {total_elapsed:.1f}s")
print(f"{'='*60}")

## Pipeline Summary

In [ ]:
import pandas as pd

summary_rows = []
for step in PIPELINE_STEPS:
    r = results.get(step["name"], {"status": "not run"})
    summary_rows.append({
        "Step": step["name"],
        "Script": step["script"],
        "Status": r["status"].upper(),
        "Time (s)": f"{r.get('time', 0):.1f}",
        "Output": step["output"],
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

n_ok = sum(1 for r in results.values() if r["status"] == "ok")
n_total = len(PIPELINE_STEPS)
print(f"\n{n_ok}/{n_total} steps completed successfully.")

## Verify Outputs

In [ ]:
import os

print("Output file verification:\n")
for step in PIPELINE_STEPS:
    p = BASE_DIR / step["output"]
    if p.exists():
        size = os.path.getsize(p)
        if p.suffix == ".csv":
            row_count = sum(1 for _ in open(p)) - 1
            print(f"  {step['output']:45s}  {size/1024:8.1f} KB  {row_count:,} rows")
        else:
            print(f"  {step['output']:45s}  {size/1024:8.1f} KB")
    else:
        print(f"  {step['output']:45s}  MISSING")

# Verify model bundle
model_path = BASE_DIR / "models" / "whale_classifier.pkl"
if model_path.exists():
    import pickle
    with open(model_path, "rb") as f:
        bundle = pickle.load(f)
    print(f"\nModel bundle keys: {list(bundle.keys())}")
    print(f"Classes: {list(bundle['label_encoder'].classes_)}")
    print(f"Features: {bundle['feature_cols']}")
else:
    print("\nModel bundle not found.")